# 신경망 모델 훈련

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # torch import 전에 설정

import copy

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

## seed / determinism

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [4]:
criterion = nn.CrossEntropyLoss()

## data

In [5]:
tr = datasets.FashionMNIST(root="./data", train=True, download=True)
train_input, train_target = tr.data.numpy(), tr.targets.numpy()

train_scaled = train_input / 255.0  # (N, 28, 28)

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

print(train_scaled.shape, train_target.shape)
print(val_scaled.shape, val_target.shape)

(48000, 28, 28) (48000,)
(12000, 28, 28) (12000,)


In [6]:
X_train = torch.tensor(train_scaled, dtype=torch.float32)
y_train = torch.tensor(train_target, dtype=torch.long)
X_val = torch.tensor(val_scaled, dtype=torch.float32)
y_val = torch.tensor(val_target, dtype=torch.long)

In [7]:
def make_train_loader(batch_size=32):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, generator=g)

val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)

## model

In [ ]:
def keras_init(m):
    # Keras Dense 기본 초기화: glorot_uniform + bias 0
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)


def model_fn(a_layer=None):
    layers = [nn.Flatten(), nn.Linear(784, 100), nn.ReLU()]
    if a_layer is not None:
        layers.append(a_layer)
    layers.append(nn.Linear(100, 10))  # softmax는 CrossEntropyLoss에 포함
    model = nn.Sequential(*layers)
    model.apply(keras_init)
    return model.to(device)


def summary(model):
    print(model)
    print("params:", sum(p.numel() for p in model.parameters()))


def keras_rmsprop(params):  # compile()에 optimizer 미지정 시 기본값
    return torch.optim.RMSprop(params, lr=1e-3, alpha=0.9, eps=1e-7)


def keras_adam(params):
    return torch.optim.Adam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-7)

## train / eval

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            n += xb.size(0)
    return total_loss / n, correct / n


def evaluate(model, loader=val_loader):
    loss, acc = run_epoch(model, loader)
    print(f"val loss: {loss:.4f} - val accuracy: {acc:.4f}")
    return loss, acc


def fit(model, optimizer, epochs=5, validation=False, callbacks=None):
    """Keras fit() 대응. history dict 반환. callbacks는 on_epoch_end(epoch, model, logs)를 가진 객체."""
    history = {"loss": [], "accuracy": []}
    if validation:
        history.update({"val_loss": [], "val_accuracy": []})
    callbacks = callbacks or []
    train_loader = make_train_loader()

    for epoch in range(epochs):
        loss, acc = run_epoch(model, train_loader, optimizer)
        history["loss"].append(loss)
        history["accuracy"].append(acc)
        logs = {"loss": loss, "accuracy": acc}
        if validation:
            v_loss, v_acc = run_epoch(model, val_loader)
            history["val_loss"].append(v_loss)
            history["val_accuracy"].append(v_acc)
            logs.update({"val_loss": v_loss, "val_accuracy": v_acc})

        stop = False
        for cb in callbacks:
            if cb.on_epoch_end(epoch, model, logs):
                stop = True
        if stop:
            break

    for cb in callbacks:
        if hasattr(cb, "on_train_end"):
            cb.on_train_end(model)
    return history

## callbacks (Keras ModelCheckpoint / EarlyStopping 대응)

In [ ]:
class ModelCheckpoint:
    def __init__(self, path, save_best_only=True):
        self.path, self.best = path, float("inf")
        self.save_best_only = save_best_only

    def on_epoch_end(self, epoch, model, logs):
        if not self.save_best_only or logs["val_loss"] < self.best:
            self.best = logs["val_loss"]
            torch.save(model.state_dict(), self.path)
        return False


class EarlyStopping:
    def __init__(self, patience=2, restore_best_weights=True):
        self.patience, self.restore = patience, restore_best_weights
        self.best, self.wait, self.best_state = float("inf"), 0, None
        self.stopped_epoch = 0

    def on_epoch_end(self, epoch, model, logs):
        if logs["val_loss"] < self.best:
            self.best, self.wait = logs["val_loss"], 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.wait += 1
        if self.wait >= self.patience:
            self.stopped_epoch = epoch
            return True
        return False

    def on_train_end(self, model):
        if self.restore and self.best_state is not None:
            model.load_state_dict(self.best_state)

## Plot History

In [ ]:
def plot_history(history, keys=("loss",), legend=None, ylabel="loss"):
    for k in keys:
        plt.plot(history[k])
    plt.xlabel("epoch")
    plt.ylabel(ylabel)
    if legend:
        plt.legend(legend)
    plt.show()

## 1) 기본 모델, 5 epoch

In [ ]:
model = model_fn()
summary(model)

history = fit(model, keras_rmsprop(model.parameters()), epochs=5)
print(history.keys())

In [ ]:
plot_history(history, ["loss"], ylabel="loss")

In [ ]:
plot_history(history, ["accuracy"], ylabel="accuracy")

## 2) 20 epoch

In [ ]:
model = model_fn()
history = fit(model, keras_rmsprop(model.parameters()), epochs=20)

In [ ]:
plot_history(history, ["loss"])

## 3) 검증 손실 함께 확인

In [ ]:
model = model_fn()
history = fit(model, keras_rmsprop(model.parameters()), epochs=20, validation=True)
print(history.keys())

In [ ]:
plot_history(history, ["loss", "val_loss"], legend=["train", "val"])

## 4) Adam

In [ ]:
model = model_fn()
history = fit(model, keras_adam(model.parameters()), epochs=20, validation=True)

In [ ]:
plot_history(history, ["loss", "val_loss"], legend=["train", "val"])

## 5) Dropout(0.3)

In [ ]:
model = model_fn(nn.Dropout(0.3))
summary(model)

In [ ]:
history = fit(model, keras_adam(model.parameters()), epochs=20, validation=True)

In [ ]:
plot_history(history, ["loss", "val_loss"], legend=["train", "val"])

## 6) 모델 저장 / 불러오기

In [ ]:
model = model_fn(nn.Dropout(0.3))
history = fit(model, keras_adam(model.parameters()), epochs=10, validation=True)

In [ ]:
torch.save(model, "model-whole.pt")                    # 모델 전체 (model.save 대응)
torch.save(model.state_dict(), "model.weights.pt")     # 가중치만 (save_weights 대응)

가중치 불러오기

In [ ]:
model = model_fn(nn.Dropout(0.3))
model.load_state_dict(torch.load("model.weights.pt", map_location=device))
model.eval()
with torch.no_grad():
    val_labels = model(X_val.to(device)).argmax(1).cpu().numpy()
print(np.mean(val_labels == val_target))

모델 전체 불러오기 (PyTorch 2.6+는 weights_only=False 필요)

In [ ]:
model = torch.load("model-whole.pt", map_location=device, weights_only=False)
evaluate(model)

## 7) ModelCheckpoint

In [ ]:
model = model_fn(nn.Dropout(0.3))
checkpoint_cb = ModelCheckpoint("best-model.pt", save_best_only=True)
fit(model, keras_adam(model.parameters()), epochs=20, validation=True, callbacks=[checkpoint_cb])

In [ ]:
model = model_fn(nn.Dropout(0.3))
model.load_state_dict(torch.load("best-model.pt", map_location=device))
evaluate(model)

## 8) ModelCheckpoint + EarlyStopping

In [ ]:
model = model_fn(nn.Dropout(0.3))
checkpoint_cb = ModelCheckpoint("best-model.pt", save_best_only=True)
early_stopping_cb = EarlyStopping(patience=2, restore_best_weights=True)
history = fit(model, keras_adam(model.parameters()), epochs=20, validation=True,
              callbacks=[checkpoint_cb, early_stopping_cb])
print(early_stopping_cb.stopped_epoch)

In [ ]:
plot_history(history, ["loss", "val_loss"], legend=["train", "val"])

In [ ]:
evaluate(model)